# VALORANT 前日試合ふり返り
ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選んでから、上から2つ実行するだけです。
ランタイムを再起動・切り替えたときも、必ずセル1 → セル2 の順に実行してください（Drive の接続とトークンの設定がセル2で行われます）。
設定は `config/default.yaml`、上書きは Drive の `valorant-replay/config.yaml` に書きます。

In [ ]:
!git -C /content/valorant-replay pull -q 2>/dev/null || git clone -q https://github.com/AK47bbddk/valorant-replay /content/valorant-replay
!pip -q install -e '/content/valorant-replay[video]'
!test -x ~/.deno/bin/deno || (curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1)  # YouTube の解読に使う JS ランタイム
!nvidia-smi -L >/dev/null 2>&1 && ! python -c "import onnxruntime as o; assert 'CUDAExecutionProvider' in o.get_available_providers()" 2>/dev/null && (pip -q uninstall -y onnxruntime; pip -q install --force-reinstall --no-deps onnxruntime-gpu || pip -q install --force-reinstall onnxruntime)  # GPU があれば OCR も GPU で（失敗したら CPU 版に戻す）
%cd /content/valorant-replay

In [ ]:
from google.colab import drive, userdata
import os, shutil
if not os.path.ismount('/content/drive'):
    if os.path.exists('/content/drive'):
        shutil.move('/content/drive', '/content/drive_local_backup')  # 未マウント時に Colab 内へできたフォルダを退避
    drive.mount('/content/drive')
def secret(k, required=False):
    try:
        v = (userdata.get(k) or '').strip()
    except Exception as e:   # 未登録・ノートブックからのアクセスがオフ・名前違い など
        if required:
            print(f'⚠ シークレット {k} を読めません: {type(e).__name__} {e}')
        return ''
    if required:
        print(f'✓ {k} を読み込みました（{len(v)} 文字, 先頭 {v[:4]}…）')
    return v
os.environ['NOTION_TOKEN'] = secret('NOTION_TOKEN', required=True)
os.environ['YT_COOKIES'] = secret('YT_COOKIES')  # 任意。Drive の valorant-replay/cookies.txt でもよい
# 任意。プレイブックの ATK/DEF summary を書かせる無料の LLM（どちらか 1 つで OK）
os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY')  # https://openrouter.ai/keys （DeepSeek V4 Flash 無料版）
os.environ['GEMINI_API_KEY'] = secret('GEMINI_API_KEY')          # https://aistudio.google.com/apikey （Gemini 無料枠）
os.environ['VREPLAY_CONFIG'] = '/content/drive/MyDrive/valorant-replay/config.yaml'  # 無ければ既定値で動く
!vreplay run   # 例: --date 2026-10-21 / --match <VLRのURL> / --no-video / --force

### プレイブック（チーム × マップ × 構成ごとのスライド）
毎日の実行（セル2）で解析結果がウェアハウス（Drive の `valorant-replay/warehouse.sqlite3`）にたまります。
下のセルの `TEAM`・`MAP` を変えて実行すると、Strategy（strong / weak point）・配置・デフォルト・セット・リテイク・ポストプラントのスライドを作って表示し、HTML をダウンロードします（印刷すると 1 スライド 1 ページの PDF）。
`--sync` は以前の実行の report.json もまとめて取り込みます（初回だけで OK）。
`NOTION_PARENT` に親ページの URL を入れると、Notion にも ATK / DEF のデータベース（型ごとのページ・例のラウンドの動画・位置の画像）つきで書きます。シークレットに `OPENROUTER_API_KEY` か `GEMINI_API_KEY`（どちらも無料）があれば、ATK / DEF summary の文章を LLM が数字から書きます。

In [ ]:
TEAM, MAP = 'G2', 'Split'   # 空文字 '' にすると全部
NOTION_PARENT = ''          # Notion にも書くなら、プレイブックを置く親ページの URL（空なら HTML だけ）
ARGS = (f'--team "{TEAM}" ' if TEAM else '') + (f'--map "{MAP}" ' if MAP else '') + (f'--notion "{NOTION_PARENT}"' if NOTION_PARENT else '')
!vreplay playbook --sync $ARGS
import glob, os
from IPython.display import HTML, display
from google.colab import files
d = '/content/drive/MyDrive/valorant-replay/playbook/'
decks = sorted(glob.glob(d + 'team_*.html'), key=os.path.getmtime, reverse=True)
for f in decks[:3]:
    print(f)
    display(HTML(open(f, encoding='utf-8').read()))
    files.download(f)

### ROI の確認（HUD の位置がずれているとき）
下のセルの URL を好きな試合に変えて実行すると、枠を描いた画像がその場に表示され、ダウンロードもされます。

※ ランタイムを切り替える・切断する前に最後のセルを実行してください（Drive への書き込みが反映されずに消えることがあります）。

In [ ]:
!vreplay calibrate --match https://www.vlr.gg/754737/loud-vs-team-vitality-valorant-champions-2026-gf --map 1 --at 600
import os
from IPython.display import Image, display
from google.colab import files
d = '/content/drive/MyDrive/valorant-replay/calibrate/'
for n in ('rois.jpg', 'panels.jpg', 'table.jpg'):
    if os.path.exists(d + n):
        display(Image(d + n))
        files.download(d + n)

In [ ]:
drive.flush_and_unmount()  # Drive への書き込みを確定（終わる前・ランタイムを変える前に）